## Example 1: A Resource with GET and PUT behind Api.add_resource

In [ ]:
import importlib.metadata
from flask import Flask, request
import flask_restful
from flask_restful import Api, Resource
from flask_restful.__version__ import __version__

print("flask-restful", __version__)
assert __version__ == importlib.metadata.version("flask-restful")

app = Flask(__name__)
api = Api(app)
todos = {}


class Todo(Resource):
    def get(self, todo_id):
        return {todo_id: todos[todo_id]}

    def put(self, todo_id):
        todos[todo_id] = request.get_json()["task"]
        return {todo_id: todos[todo_id]}, 201


api.add_resource(Todo, "/todos/<string:todo_id>")
client = app.test_client()
created = client.put("/todos/first", json={"task": "write the notebook"})
print("PUT ->", created.status_code, created.get_json())
assert created.status_code == 201 and created.get_json() == {"first": "write the notebook"}
fetched = client.get("/todos/first")
print("GET ->", fetched.status_code, fetched.get_json(), fetched.content_type)
assert fetched.status_code == 200 and fetched.get_json() == {"first": "write the notebook"}
assert fetched.content_type == "application/json"  # a returned dict is serialised as JSON
# A method the Resource does not define is answered with 405, not a server error.
assert client.delete("/todos/first").status_code == 405
print("OK -- a Resource's get/put methods answer GET/PUT; DELETE is 405 Method Not Allowed")

## Example 2: Parse and validate request arguments with RequestParser

In [ ]:
from flask import Flask
from flask_restful import Api, Resource, reqparse

app = Flask(__name__)
api = Api(app)

parser = reqparse.RequestParser()
parser.add_argument("name", type=str, required=True, help="name is required")
parser.add_argument("age", type=int, location="json")
parser.add_argument("role", choices=("admin", "user"), default="user", location="json")


class Users(Resource):
    def post(self):
        args = parser.parse_args()
        return dict(args), 201


api.add_resource(Users, "/users")
client = app.test_client()
ok = client.post("/users", json={"name": "Ada", "age": "36"})
print("valid ->", ok.status_code, ok.get_json())
assert ok.status_code == 201 and ok.get_json() == {"name": "Ada", "age": 36, "role": "user"}  # "36" became 36

missing = client.post("/users", json={"age": 3})
print("no name ->", missing.status_code, missing.get_json())
assert missing.status_code == 400 and "name is required" in missing.get_json()["message"]["name"]

bad_choice = client.post("/users", json={"name": "Eve", "role": "root"})
print("bad role ->", bad_choice.status_code, bad_choice.get_json())
assert bad_choice.status_code == 400 and "role" in bad_choice.get_json()["message"]
print("OK -- RequestParser converts types, fills defaults and refuses missing or invalid arguments with 400")

## Example 3: Shape the output with fields and marshal()

In [ ]:
from datetime import datetime, timezone
from flask_restful import fields, marshal

book_fields = {
    "title": fields.String,
    "pages": fields.Integer,
    "price": fields.Float,
    "published": fields.DateTime(dt_format="iso8601"),
    "author": fields.Nested({"name": fields.String, "country": fields.String(default="unknown")}),
    "tags": fields.List(fields.String),
    "isbn": fields.String(attribute="identifiers.isbn"),
}
book = {
    "title": "Example Book",
    "pages": "320",
    "price": 12.5,
    "published": datetime(2024, 5, 1, 9, 30, tzinfo=timezone.utc),
    "author": {"name": "A. Writer"},
    "tags": ["fiction", "classic"],
    "identifiers": {"isbn": "978-0-00-000000-0"},
    "internal_note": "not part of the output",
}
output = marshal(book, book_fields)
for key, value in output.items():
    print(f"  {key}: {value!r}")
assert list(output) == list(book_fields)  # only the declared fields, in declared order
assert output["pages"] == 320 and output["price"] == 12.5
assert output["published"] == "2024-05-01T09:30:00+00:00", output["published"]
assert output["author"] == {"name": "A. Writer", "country": "unknown"}
assert output["tags"] == ["fiction", "classic"] and output["isbn"] == "978-0-00-000000-0"
assert "internal_note" not in output
print("OK -- marshal() keeps only the declared fields, converts their types and fills defaults")

## Example 4: marshal_with, abort() and (body, status, headers) returns

In [ ]:
from flask import Flask
from flask_restful import Api, Resource, abort, fields, marshal_with

app = Flask(__name__)
api = Api(app)
items = {1: {"id": 1, "name": "lamp", "stock": 4}}
item_fields = {"id": fields.Integer, "name": fields.String}


class Item(Resource):
    @marshal_with(item_fields)
    def get(self, item_id):
        if item_id not in items:
            abort(404, message=f"item {item_id} does not exist")
        return items[item_id]


class Echo(Resource):
    def get(self):
        return {"echo": True}, 202, {"X-Handled-By": "flask-restful"}


api.add_resource(Item, "/items/<int:item_id>")
api.add_resource(Echo, "/echo")
client = app.test_client()
found = client.get("/items/1")
missing = client.get("/items/99")
echo = client.get("/echo")
print("found ->", found.status_code, found.get_json())
print("missing ->", missing.status_code, missing.get_json())
print("echo ->", echo.status_code, echo.get_json(), echo.headers.get("X-Handled-By"))
assert found.status_code == 200 and found.get_json() == {"id": 1, "name": "lamp"}  # "stock" is not marshalled
assert missing.status_code == 404 and missing.get_json()["message"] == "item 99 does not exist"
assert echo.status_code == 202 and echo.headers["X-Handled-By"] == "flask-restful"
print("OK -- marshal_with filters the output, abort() returns a JSON error, a (body, status, headers) tuple is honoured")

## Example 5: A custom CSV representation chosen by the Accept header

In [ ]:
import csv, io
from flask import Flask, make_response
from flask_restful import Api, Resource

app = Flask(__name__)
api = Api(app)


@api.representation("text/csv")
def output_csv(data, code, headers=None):
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=list(data[0]))
    writer.writeheader()
    writer.writerows(data)
    response = make_response(buffer.getvalue(), code)
    response.headers.extend(headers or {})
    return response


class Scores(Resource):
    def get(self):
        return [{"player": "ana", "score": 12}, {"player": "bo", "score": 9}]


api.add_resource(Scores, "/scores")
client = app.test_client()
as_json = client.get("/scores", headers={"Accept": "application/json"})
as_csv = client.get("/scores", headers={"Accept": "text/csv"})
print("JSON:", as_json.content_type, as_json.get_json())
print("CSV:", as_csv.content_type, repr(as_csv.get_data(as_text=True)))
assert as_json.get_json() == [{"player": "ana", "score": 12}, {"player": "bo", "score": 9}]
assert as_csv.content_type.startswith("text/csv")
assert as_csv.get_data(as_text=True).splitlines() == ["player,score", "ana,12", "bo,9"]
# An Accept type with no representation falls back to the default, JSON.
fallback = client.get("/scores", headers={"Accept": "application/xml"})
assert fallback.status_code == 200 and fallback.content_type == "application/json", fallback.content_type
print("OK -- the same Resource answers JSON or CSV depending on the Accept header")